展示了如何使用 minGPT 或 huggingface/transformers，在给定提示词和一些超参数的情况下生成文本。

In [1]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from mingpt.model import GPT
from mingpt.utils import set_seed
from mingpt.bpe import BPETokenizer
set_seed(3407)

In [2]:
use_mingpt = True # 使用 minGPT 还是 huggingface/transformers 模型？
model_type = 'gpt2-xl'
device = 'cuda'

In [3]:
if use_mingpt:
    model = GPT.from_pretrained(model_type)
else:
    model = GPT2LMHeadModel.from_pretrained(model_type)
    model.config.pad_token_id = model.config.eos_token_id # 消除警告

# 将模型移动到设备并设置为评估模式
model.to(device)
model.eval();

number of parameters: 1557.61M


In [4]:

def generate(prompt='', num_samples=10, steps=20, do_sample=True):
        
    # 将输入提示词分词为整数输入序列
    if use_mingpt:
        tokenizer = BPETokenizer()
        if prompt == '':
            # 为了创建无条件样本...
            # 手动创建一个仅包含特殊 <|endoftext|> 标记的张量
            # 类似于 OpenAI 在此处的代码实现：https://github.com/openai/gpt-2/blob/master/src/generate_unconditional_samples.py
            x = torch.tensor([[tokenizer.encoder.encoder['<|endoftext|>']]], dtype=torch.long)
        else:
            x = tokenizer(prompt).to(device)
    else:
        tokenizer = GPT2Tokenizer.from_pretrained(model_type)
        if prompt == '': 
            # 为了创建无条件样本...
            # huggingface/transformers 分词器对这些字符串进行了特殊处理
            prompt = '<|endoftext|>'
        encoded_input = tokenizer(prompt, return_tensors='pt').to(device)
        x = encoded_input['input_ids']
    
    # 我们将批量处理所有需要的样本数量，因此展开批次维度
    x = x.expand(num_samples, -1)

    # 在一个批次中多次前向传播模型以获取样本
    y = model.generate(x, max_new_tokens=steps, do_sample=do_sample, top_k=40)
    
    for i in range(num_samples):
        out = tokenizer.decode(y[i].cpu().squeeze())
        print('-'*80)
        print(out)
        

In [5]:
generate(prompt='Andrej Karpathy, the', num_samples=10, steps=20)

--------------------------------------------------------------------------------
Andrej Karpathy, the chief of the criminal investigation department, said during a news conference, "We still have a lot of
--------------------------------------------------------------------------------
Andrej Karpathy, the man whom most of America believes is the architect of the current financial crisis. He runs the National Council
--------------------------------------------------------------------------------
Andrej Karpathy, the head of the Department for Regional Reform of Bulgaria and an MP in the centre-right GERB party
--------------------------------------------------------------------------------
Andrej Karpathy, the former head of the World Bank's IMF department, who worked closely with the IMF. The IMF had
--------------------------------------------------------------------------------
Andrej Karpathy, the vice president for innovation and research at Citi who oversaw the team's work to mak